In [ ]:
import polars as pl
from pathlib import Path

csv_path = Path("../data/raw/lcdv2/hourly/2025/72662604864-2025.csv")

df = pl.read_csv(
    csv_path,
    infer_schema_length=0,   # force all Utf8
    schema_overrides={"DATE": pl.Utf8, "STATION": pl.Utf8}
)

# Convert DATE to a datetime type
df = (
    df
    .with_columns([
        pl.col("DATE").str.strptime(pl.Datetime, strict=False).alias("dt"),
    ])
)

In [2]:
# Identify hourly fields
hourly_fields = [c for c in df.columns if c.startswith("Hourly")]
hourly_fields

['HourlyAltimeterSetting',
 'HourlyDewPointTemperature',
 'HourlyDryBulbTemperature',
 'HourlyPrecipitation',
 'HourlyPresentWeatherType',
 'HourlyPressureChange',
 'HourlyPressureTendency',
 'HourlyRelativeHumidity',
 'HourlySkyConditions',
 'HourlySeaLevelPressure',
 'HourlyStationPressure',
 'HourlyVisibility',
 'HourlyWetBulbTemperature',
 'HourlyWindDirection',
 'HourlyWindGustSpeed',
 'HourlyWindSpeed']

In [6]:
# Build population stats for each hourly field
hourly_stats = (
    pl.DataFrame({
        "field": hourly_fields,
        "non_empty": [
            df.filter(pl.col(f) != "").height for f in hourly_fields
        ],
        "empty": [
            df.filter(pl.col(f) == "").height for f in hourly_fields
        ],
    })
    .with_columns([
        (pl.col("non_empty") / (pl.col("non_empty") + pl.col("empty")) * 100)
        .alias("pct_populated")
    ])
    .sort("pct_populated")
)


print(hourly_stats)

shape: (16, 4)
┌───────────────────────────┬───────────┬───────┬───────────────┐
│ field                     ┆ non_empty ┆ empty ┆ pct_populated │
│ ---                       ┆ ---       ┆ ---   ┆ ---           │
│ str                       ┆ i64       ┆ i64   ┆ f64           │
╞═══════════════════════════╪═══════════╪═══════╪═══════════════╡
│ HourlyPressureChange      ┆ 0         ┆ 17176 ┆ 0.0           │
│ HourlyPressureTendency    ┆ 0         ┆ 17176 ┆ 0.0           │
│ HourlySeaLevelPressure    ┆ 0         ┆ 17176 ┆ 0.0           │
│ HourlyPrecipitation       ┆ 585       ┆ 16591 ┆ 3.405915      │
│ HourlyPresentWeatherType  ┆ 2483      ┆ 14693 ┆ 14.456218     │
│ HourlyWindGustSpeed       ┆ 2962      ┆ 14214 ┆ 17.244993     │
│ HourlySkyConditions       ┆ 15721     ┆ 1455  ┆ 91.528878     │
│ HourlyWetBulbTemperature  ┆ 16628     ┆ 548   ┆ 96.809502     │
│ HourlyStationPressure     ┆ 16630     ┆ 546   ┆ 96.821146     │
│ HourlyVisibility          ┆ 16944     ┆ 232   ┆ 98.649278  

In [7]:
# Peek at example values for each hourly field
examples = {
    f: df.select(pl.col(f)).filter(pl.col(f) != "").head(5)
    for f in hourly_fields
}

examples

{'HourlyAltimeterSetting': shape: (5, 1)
 ┌────────────────────────┐
 │ HourlyAltimeterSetting │
 │ ---                    │
 │ str                    │
 ╞════════════════════════╡
 │ 29.87                  │
 │ 29.87                  │
 │ 29.87                  │
 │ 29.87                  │
 │ 29.88                  │
 └────────────────────────┘,
 'HourlyDewPointTemperature': shape: (5, 1)
 ┌───────────────────────────┐
 │ HourlyDewPointTemperature │
 │ ---                       │
 │ str                       │
 ╞═══════════════════════════╡
 │ 22                        │
 │ 22                        │
 │ 22                        │
 │ 22                        │
 │ 21                        │
 └───────────────────────────┘,
 'HourlyDryBulbTemperature': shape: (5, 1)
 ┌──────────────────────────┐
 │ HourlyDryBulbTemperature │
 │ ---                      │
 │ str                      │
 ╞══════════════════════════╡
 │ 26                       │
 │ 26                       │
 │ 26      